In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Load the datasets
ling = pd.read_csv("Ling.csv")
nazario = pd.read_csv("Nazario.csv")
nigerian = pd.read_csv("Nigerian_Fraud.csv")

# 2. Clean missing values and combine Subject + Body into a single text feature
def prepare_data(df):
    df['subject'] = df['subject'].fillna("")
    df['body'] = df['body'].fillna("")
    df['text'] = df['subject'] + " " + df['body']
    return df[['text', 'label']]

ling_clean = prepare_data(ling)
nazario_clean = prepare_data(nazario)
nigerian_clean = prepare_data(nigerian)

# 3. Merge all datasets into one combined dataset
final_data = pd.concat([ling_clean, nazario_clean, nigerian_clean], ignore_index=True)
print(f"Total merged records: {len(final_data)}")

# 4. Split data into training (80%) and testing (20%) sets
X = final_data['text']
y = final_data['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Convert text to numerical features (TF-IDF Vectorization)
# We limit to 5000 features to optimize Colab's RAM usage
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# 6. Train the Random Forest Classifier
print("Training the Random Forest model...")
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train_vectorized, y_train)

# 7. Evaluate and print results
predictions = model.predict(X_test_vectorized)
print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, predictions))

Total merged records: 7756
Training the Random Forest model...

--- Model Evaluation ---
Accuracy: 99.55%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       503
           1       1.00      1.00      1.00      1049

    accuracy                           1.00      1552
   macro avg       1.00      0.99      0.99      1552
weighted avg       1.00      1.00      1.00      1552



In [2]:
# Paste a suspicious SMS or Email text here
sample_text = ["URGENT: Your bank account is locked. Click here to verify your identity immediately!"]

# Transform and predict
sample_vectorized = vectorizer.transform(sample_text)
prediction = model.predict(sample_vectorized)

if prediction[0] == 1:
    print("Risk Level: High (Scam/Phishing Detected)")
else:
    print("Risk Level: Low (Legitimate)")

Risk Level: High (Scam/Phishing Detected)
